In [ ]:
import jax
import pandas as pd

jax.config.update("jax_platform_name", "cpu")

from gould_2026.estimator import Pipeline, ArrayWithTime, CenteringEstimator
from gould_2026.utils import angle_between
from gould_2026.dimension_reduction.prosvd import proSVD
from gould_2026.prediction.kalman_filter import StreamingKalmanFilter
from gould_2026.stim_regressor import StimRegressor, StimAutoReg
from gould_2026.datasets import Naumann24uDataset
from gould_2026.plotting import Palette, LINEWIDTH, paper_plot_context
from scipy.stats import linregress
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import mantel_test
import pandas as pd
import seaborn as sns
from scipy.stats import wilcoxon
from tqdm.notebook import tqdm

rng = np.random.default_rng(42)


In [ ]:
output1 = None
output2 = None
output3 = None
output4 = None
output5 = None
output6 = None
output7_response_pairwise_plot = None
output8_quiver_plot = None


In [ ]:
rng = np.random.default_rng()
d = Naumann24uDataset(1)

target_neurons = np.unique(d.opto_stimulations.target_neuron)

d.neural_data[np.isnan(d.neural_data)] = 0

def target_neuron_to_vector(tn):
    v = target_neurons * 0
    v[target_neurons == tn] = 1
    return v

d.opto_stimulations['stim_vector'] = d.opto_stimulations['target_neuron'].apply(target_neuron_to_vector)




In [ ]:
d.opto_stimulations

In [ ]:
p = Pipeline([
    CenteringEstimator(init_size=100, nan_when_uninitialized=True),
    proSVD(k=10),
])


latents = p.offline_run_on(d.neural_data, show_tqdm=True)
stim = ArrayWithTime(np.squeeze([x for x  in d.opto_stimulations['stim_vector']]), d.opto_stimulations['time'])


In [ ]:
%matplotlib inline

fig, axs = plt.subplots(nrows=2, figsize=(10,6), layout='constrained', sharex=True)
axs[0].plot(latents.t, latents, '.-')


kf = StreamingKalmanFilter(log_level=2, check_dt=True)
kf.offline_run_on(latents)
errors = ArrayWithTime.from_list(kf.log['pred_error'], squeeze_type='to_2d')
errors  = ArrayWithTime(np.linalg.norm(errors, axis=1), errors.t)
axs[1].plot(errors.t, errors, '.-')

stim_ts = []
for t1, t2 in zip(stim.t, stim.t[1:]):
    s = errors.slice_by_time(slice(t1, t2))
    stim_t = s.t[np.argmax(s)]
    stim_ts.append(max(stim_t - 2 * latents.dt, t1))

for t in stim.t:
    axs[0].axvline(t, color='red', alpha=0.3)
    axs[1].axvline(t, color='red', alpha=0.3)

for t in stim_ts:
    # axs[0].axvline(t, color='blue', alpha=0.3)
    axs[1].axvline(t, color='blue', alpha=0.3)

stim_shifted = ArrayWithTime(stim[:-1], np.array(stim_ts))

In [ ]:
srs = {
    'blind': StimRegressor(log_level=2, heed_stimuli=False, attempt_correction=False),
    'reg': StimRegressor(log_level=2, stim_delay=1*latents.dt),
}
srs['reg'].stim_autoreg = StimAutoReg(n_steps_to_consider=4)

for sr in srs.values():
    sr.offline_run_on([(stim_shifted, 'stim'), (latents, 'X')], show_tqdm=True)

In [ ]:
%matplotlib inline
fig, ax = plt.subplots(constrained_layout=True, figsize=(10,4))

stim_df = pd.DataFrame()
for k, sr in srs.items():
    error = ArrayWithTime.from_list(sr.log['pred_error'], squeeze_type='to_2d')
    norm_error = ArrayWithTime(np.linalg.norm(error, axis=1),error.t)
    ax.plot(norm_error.t, norm_error, '.-', label=f'{k} mse_whole={np.nanmean(norm_error ** 2):.2f} mse[600:]={np.nanmean(norm_error.slice_by_time(slice(600,None))** 2):.2f}')

    pre_df = []
    for s in stim_shifted:
        i = norm_error.time_to_sample(s.t)
        e = norm_error.slice(slice(i, i+6))

        pre_df.append({
            'norm_error':e,
            'stim':s,
            't':s.t,
            'group': k,
            'stim_i': i
        })
    stim_df = pd.concat([stim_df, pd.DataFrame(pre_df)], ignore_index=True)

stim_df['error'] = stim_df['norm_error'].apply(lambda x: np.sqrt(np.mean(x**2)))
stim_df['target'] = stim_df['stim'].apply(np.argmax).astype('category')
ax.legend()
# ax.set_xlim([700, 1300])
# ax.set_ylim([0,12])

ax.set_xlabel('Time (s)')
ax.set_ylabel('norm error')

for t in stim_shifted.t:
    ax.axvline(t, color='red', alpha=0.3)

In [ ]:
fig, ax = plt.subplots(constrained_layout=True, figsize=(10,4))

for c, (k, sr) in zip([Palette.blind, Palette.stim_regressed], srs.items()):
    error = ArrayWithTime.from_list(sr.log['pred_error'], squeeze_type='to_2d')
    norm_error = ArrayWithTime(np.linalg.norm(error, axis=1),error.t)
    ax.plot(norm_error.t, norm_error, '-', color=c, label=f'{k} mse_whole={np.nanmean(norm_error ** 2):.2f} mse[600:]={np.nanmean(norm_error.slice_by_time(slice(600,None))** 2):.2f}')


stim_df['error'] = stim_df['norm_error'].apply(lambda x: np.sqrt(np.mean(x**2)))
stim_df['target'] = stim_df['stim'].apply(np.argmax).astype('category')
# ax.legend()
# ax.set_xlim([700, 1300])
ax.set_ylim([0,12])

ax.set_xlabel('Time (s)')
ax.set_ylabel('norm error')




if output1 is not None:
    fig.savefig(output1)


In [ ]:

from scipy.spatial.distance import pdist, squareform
sreg = srs['reg'].stim_reg
m = squareform(pdist(sreg.output_history[:sreg.n_observed], metric=lambda x,y: angle_between(x,y)))
plt.matshow(m, cmap='plasma', vmin=0, vmax=150)

if output7_response_pairwise_plot is not None:
    plt.savefig(output7_response_pairwise_plot)


In [ ]:
fig, ax = plt.subplots(constrained_layout=True, figsize=(5,5))
sns.stripplot(stim_df, x='group', y='error', ax=ax)

pivot = stim_df.pivot(index='stim_i', columns='group', values='error').dropna()
print(f"{pivot['blind'].median() = }")
print(f"{pivot['reg'].median() = }")
wilcoxon(pivot['blind'], pivot['reg'])

if output6 is not None:
    fig.savefig(output6)


In [ ]:
(pivot['blind'] - pivot['reg']).median()

In [ ]:
sns.lineplot(stim_df, x='group', y='error', hue='t')

In [ ]:

sns.scatterplot(stim_df, x='t', y='error', hue='group')

In [ ]:
fig, ax = plt.subplots(constrained_layout=True, figsize=(5,5))
pivot = stim_df.pivot(index='stim_i', columns='group', values='error')
sns.scatterplot(pivot, x='blind', y='reg', ax=ax)
ax.axis('equal')

lims = [
    min(ax.get_xlim()[0], ax.get_ylim()[0]),
    max(ax.get_xlim()[1], ax.get_ylim()[1]),
]
ax.plot(lims, lims, '--', color='gray', zorder=0)

In [ ]:
kf_error = ArrayWithTime.from_list(srs['reg'].log['pred_error'])

fig, ax = plt.subplots(constrained_layout=True, figsize=(8,8))

preq_errors = ArrayWithTime.from_list(srs['reg'].stim_reg.log['preq_errors'], squeeze_type='to_2d')
preq_errors = np.linalg.norm(preq_errors,axis=1)
n_observed = srs['reg'].stim_reg.n_observed


ax.scatter(latents[:,0], latents[:,1], c='gray', alpha=0.5, s=1)
error_scatter = ax.scatter(srs['reg'].stim_reg.input_histories[0][:n_observed,0], srs['reg'].stim_reg.input_histories[0][:n_observed,1], c=preq_errors, cmap='plasma')
ax.set_xlim(right=20)
fig.colorbar(error_scatter)


if output2 is not None:
    fig.savefig(output2)



In [ ]:
%matplotlib inline
fig, axs = plt.subplots(constrained_layout=True, figsize=np.array([10,5])/3, ncols=2)

n = 25
l = 1
r = 7
time_slice = slice(stim_shifted.t[n] -l, stim_shifted.t[n] + r)

ax = axs[1]
ax.scatter(latents[:,0], latents[:,1], c='gray', alpha=0.5, s=1)
# ax.plot(latents[:,0], latents[:,1], '-')
ax.set_xlim(right=20)
idx = np.argmin(np.linalg.norm(latents - srs['reg'].stim_reg.input_histories[0][0],axis=1))
assert np.linalg.norm(latents.slice(idx) - srs['reg'].stim_reg.input_histories[0][0]) < 1e-10
idx = latents.time_to_sample(srs['reg'].stim_reg.input_histories[2][:n_observed, 0]) -1
latents_at_stims = latents.slice(idx).slice_by_time(time_slice)
ax.scatter(latents_at_stims[:,0], latents_at_stims[:,1], c='r')
latent_slice = latents.slice_by_time(time_slice)
ax.plot(latent_slice[:,0], latent_slice[:,1])

ax = axs[0]
# full_slice = d.neural_data.slice_by_time(time_slice)
# ax.plot(full_slice.t, full_slice, 'k');
ax.plot(latent_slice.t, latent_slice, 'k');
for t in latents_at_stims.t:
    ax.axvline(t, color='r', alpha=0.5)
ax.set_xlabel('Time (s)')


if output3 is not None:
    fig.savefig(output3)



In [ ]:
%matplotlib inline
fig, ax = plt.subplots(figsize=np.array([8, 6]), constrained_layout=True)

ax.scatter(latents[:,0], latents[:,1], c='gray', alpha=0.5, s=LINEWIDTH* 1/1.5)
ax.set_xlim(right=20)

idx = np.argmin(np.linalg.norm(latents - srs['reg'].stim_reg.input_histories[0][0],axis=1))
assert np.linalg.norm(latents.slice(idx) - srs['reg'].stim_reg.input_histories[0][0]) < 1e-10
idx = latents.time_to_sample(srs['reg'].stim_reg.input_histories[2][:n_observed, 0]) -1

pre_points = latents.slice(idx)
latent_deltas = latents.slice(idx + 4) - latents.slice(idx)

for pre_point, delta, neuron in zip(pre_points, latent_deltas, d.opto_stimulations.target_neuron):
    ax.quiver(pre_point[0], pre_point[1], delta[0], delta[1], angles='xy', scale_units='xy', scale=1.5, width=.004, color=f'C{neuron}')

if output8_quiver_plot is not None:
    fig.savefig(output8_quiver_plot)

ax.set_xticks([])
ax.set_yticks([])



In [ ]:
idx = np.argmin(np.linalg.norm(latents - srs['reg'].stim_reg.input_histories[0][0],axis=1))
idx

In [ ]:
%matplotlib inline
fig, ax = plt.subplots(figsize=np.array([8, 6]), constrained_layout=True)
ax.scatter(latents[:,0], latents[:,1], alpha=0.5, s=1, color='gray')
ax.set_xlim(right=20)


angles = np.atan2(latent_deltas[:,1], latent_deltas[:,0])
s = np.linalg.norm(latent_deltas, axis=1)
# ax.scatter(latent_deltas[:,0] + pre_points[:,0], latent_deltas[:,1] +pre_points[:,1], c=angles, s=s)
ax.scatter(pre_points[:,0], pre_points[:,1], c=angles, s=s)


ax.set_xlim(right=20)



In [ ]:
fig, ax = plt.subplots(figsize=np.array([1, 1])*4, constrained_layout=True)

idx = np.linalg.norm(latent_deltas, axis=1) > 10

axis_slice = slice(None)

distances = pdist(pre_points[idx,:][:,axis_slice], metric='mahalanobis')
angles = pdist(latent_deltas[idx,:][:,axis_slice], metric=angle_between)
stim_distances = pdist(np.array(d.opto_stimulations.target_neuron)[:-1,None][idx], 'hamming')

x, y, slope, intercept, r = mantel_test.single_fit(
    x=distances,
    y=angles,
    confound_x=stim_distances,
    rng=rng,
    shuffle=False,
)

ax.scatter(x, y, s=1)

x = np.array(ax.get_xlim())
ax.plot(x, x*slope + intercept)


In [ ]:
# partial mantel test

test_statistic, null_samples, p = mantel_test.mantel_test(
    x=distances,
    y=angles,
    confound_x=stim_distances,
    rng=rng,
    n=100_000,
    use_tqdm=True,
)

fig, ax = plt.subplots(figsize=np.array([4, 3]), constrained_layout=True)
ax.hist(null_samples, label='null', bins=100, density=True)
ax.axvline(test_statistic, color='r')
ax.set_title(f'Fish partial mantel test (p$\\approx${p:.4f})')
ax.set_xlabel('partial regression r')
ax.set_ylabel('density')


In [ ]:
plt.plot(stim_shifted.t + latents.dt - srs['reg'].stim_reg.input_histories[2][:n_observed, 0])
# TODO: resolve this!!